# Khaledoon — OmniVoice Unified Worker
عامل توليد واحد لكل الحكايات: كارما والراوي مزوّدان منفصلان داخل نفس المحرك.
العامل **لا يقرر ماذا يُولَّد**؛ ينفّذ فقط الطلبات التي ضغطتَ توليدها من صفحة المعاينة، مقطعًا واحدًا في كل مرة، ويرفع الناتج إلى التخزين الخاص للمراجعة. لا نشر تلقائي.

**قبل التشغيل:** احفظ في Colab Secrets (مع تفعيل Notebook access): `KHALEDOON_WORKER_TOKEN` فقط. نص مرجع كارما ثابت داخل الدفتر ومتحقق من بصمته. للراوي (اختياري): `KHALEDOON_NARRATOR_REF_AUDIO_PATH` و`KHALEDOON_NARRATOR_REF_TEXT`.

In [ ]:
%pip -q install git+https://github.com/k2-fsa/OmniVoice.git soundfile requests numpy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/engmohamedgamalsaad-oss/nourdeen-october-1973.git'
ENGINE_BRANCH = 'main'
if not os.path.isdir('/content/khaledoon'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', ENGINE_BRANCH, REPO_URL, '/content/khaledoon'], check=True)
sys.path.insert(0, '/content/khaledoon/scripts')
import omnivoice_engine
print('Engine loaded from', omnivoice_engine.__file__)

In [ ]:
import uuid
from google.colab import drive, userdata
drive.mount('/content/drive')

WORKER_URL = 'https://ypwprddkscnfblhftkeb.supabase.co/functions/v1/story-audio-worker'
KARMA_REF_AUDIO_PATH = '/content/drive/MyDrive/Karma_Voice/assets_audio_science_magdi-yacoub_ar_000.mp3'

def secret(name):
    try:
        return str(userdata.get(name) or '').strip()
    except Exception:
        return ''

WORKER_TOKEN = secret('KHALEDOON_WORKER_TOKEN')
KARMA_REF_TEXT = "أنا كارما، ولما سمعت حكاية الدكتور مجدي يعقوب، فكرت في حاجة مهمة: إزاي طفل بيحب يساعد الناس ممكن يكبر ويبقى طبيبًا ينقذ حياة أطفال كتير؟"
KARMA_REF_TEXT_SHA256 = '50cce1612e8f55127d729c355b964e9d5335b21382572da3d96245ca37013346'
import hashlib
if hashlib.sha256(KARMA_REF_TEXT.encode('utf-8')).hexdigest() != KARMA_REF_TEXT_SHA256:
    raise ValueError('Karma reference transcript integrity check failed.')
NARRATOR_REF_AUDIO_PATH = secret('KHALEDOON_NARRATOR_REF_AUDIO_PATH')
NARRATOR_REF_TEXT = secret('KHALEDOON_NARRATOR_REF_TEXT')
if not WORKER_TOKEN:
    raise ValueError('احفظ KHALEDOON_WORKER_TOKEN في Colab Secrets وفعّل Notebook access.')
WORKER_ID = 'colab-' + str(uuid.uuid4())[:8]
print('Config ready:', WORKER_ID)

In [ ]:
import torch
from omnivoice import OmniVoice
from omnivoice_engine import MODEL_ID
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if device == 'cpu':
    print('تحذير: الجهاز CPU. اختر Runtime ثم Change runtime type ثم GPU.')
voice_model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
print('Model loaded on', device)

In [ ]:
from omnivoice_engine import KarmaProvider, NarratorProvider, ProviderRegistry, WorkerClient

registry = ProviderRegistry()
registry.register(KarmaProvider(KARMA_REF_AUDIO_PATH, KARMA_REF_TEXT))
if NARRATOR_REF_AUDIO_PATH and NARRATOR_REF_TEXT:
    registry.register(NarratorProvider(NARRATOR_REF_AUDIO_PATH, NARRATOR_REF_TEXT))
    print('Providers: karma + narrator')
else:
    print('Providers: karma only (narrator reference not configured; narrator jobs will be released back to the queue).')
client = WorkerClient(WORKER_URL, WORKER_TOKEN, WORKER_ID)

In [ ]:
# يعمل حتى توقف الخلية يدويًا. أثناء الانتظار يظهر العامل «متصلًا» في صفحة المعاينة.
from omnivoice_engine import run_worker
run_worker(client, voice_model, registry, poll_seconds=20)